# Triton 研究项目：Colab 远程验收入口

GPU 草案未在 Mac 验证。本 notebook 不包含另一套算子实现。先备份结果再断开运行时。
本地交付期间不运行本 notebook。选择兼容的 NVIDIA GPU，优先 L4/A100；实际配额与可用性以账号为准。

## 1. 指定源码版本与持久化输出

先把项目以私有仓库或源码包放到可访问位置。仓库 URL 禁止包含令牌；凭据使用 Colab Secrets 或交互式认证。以下变量必须填入真实值。PERSISTENT_OUTPUT 必须指向已挂载且已验证可持久化的目录。

In [ ]:
from pathlib import Path
import subprocess
import sys
import os

REPO_URL = ""  # Private repository URL, without embedded secrets
COMMIT = ""    # Full 40-character commit SHA
PERSISTENT_OUTPUT = ""  # e.g. an already mounted persistent directory
assert REPO_URL and len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT.lower())
assert PERSISTENT_OUTPUT and Path(PERSISTENT_OUTPUT).is_dir()
PROJECT = Path("/content/triton-ops-lab")
assert not PROJECT.exists(), "Use a fresh checkout directory or inspect the existing checkout first"
subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=PROJECT, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True).strip() == COMMIT
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.12.15"], check=True)
os.environ["TRITON_LAB_OUTPUT_ROOT"] = PERSISTENT_OUTPUT
subprocess.run(["uv", "sync", "--frozen"], cwd=PROJECT, check=True)


## 2. 检查版本、设备与 GPU 正确性

运行 uv 管理的项目解释器，避免使用 Colab 预装的另一套 PyTorch。正确性失败时停止；不要为了跑通自动升级依赖或放宽阈值。

In [ ]:
def run(*args):
    subprocess.run(["uv", "run", "--frozen", *args], cwd=PROJECT, check=True)
run("triton-lab", "doctor")
run("triton-lab", "check", "--device", "cuda")
run("python", "scripts/gpu_tools.py", "check")


## 3. 小规模基准与完整实验

先 smoke，再 tune/eval。每次运行都内置 smoke 正确性门禁。不同配置预算会消耗不同算力；建议分次运行，先观察实际耗时与 CU 消耗。

In [ ]:
run("triton-lab", "bench", "--config", "configs/smoke.json")


In [ ]:
# Run only after reviewing smoke correctness and cost:
# run("triton-lab", "bench", "--config", "configs/tune.json")
# run("triton-lab", "bench", "--config", "configs/eval.json")


## 4. 分析、导出与释放

将实际生成的运行目录填入下方。确认 metadata.json、metrics.csv 与报告存在于持久化目录，并另行下载备份，再通过 Colab 菜单断开并删除运行时。停止 notebook 单元格不一定停止资源消耗。

In [ ]:
RUN_DIRECTORY = ""
assert RUN_DIRECTORY and Path(RUN_DIRECTORY).is_dir()
run("triton-lab", "analyze", RUN_DIRECTORY)
for filename in ("metadata.json", "metrics.csv"):
    assert (Path(RUN_DIRECTORY)/filename).stat().st_size > 0
